In [2]:
import pandas as pd
import sqlite3

In [3]:
df = pd.read_csv('Electric_Vehicle_Title_and_Registration_Activity_20260811.csv')

In [5]:
print('Shape:', df.shape)

Shape: (1809203, 33)


In [6]:
df.head()

,Clean Alternative Fuel Vehicle Type,VIN (1-10),DOL Vehicle ID,Model Year,Make,Model,Primary Use,Electric Range,Odometer Reading,Odometer Reading Description,...,Meets 2019 HB 2042 Sale Price/Value Requirement,2019 HB 2042: Battery Range Requirement,2019 HB 2042: Purchase Date Requirement,2019 HB 2042: Sale Price/Value Requirement,Electric Vehicle Fee Paid,Transportation Electrification Fee Paid,Hybrid Vehicle Electrification Fee Paid,2020 GEOID,Legislative District,Electric Utility
0,Battery Electric Vehicle (BEV),5YJ3E1EA9M,182320113,2021,TESLA,Model 3,Passenger,0.0,15,Actual Mileage,...,True,Battery range requirement is met,Purchase date requirement is met,Sale price/value requirement is met,Not Applicable,Not Applicable,Not Applicable,5.303303e+10,45.0,PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA)
1,Plug-in Hybrid Electric Vehicle (PHEV),KNDRJDJH5S,276871381,2025,KIA,Sorento,Passenger,30.0,0,Odometer reading is not collected at time of r...,...,False,Battery range requirement is met,This transaction type is not eligible for the ...,This transaction type is not eligible for the ...,No,No,No,5.303509e+10,23.0,PUGET SOUND ENERGY INC
2,Plug-in Hybrid Electric Vehicle (PHEV),KNDRJDJH5S,276871381,2025,KIA,Sorento,Passenger,30.0,42,Actual Mileage,...,False,Battery range requirement is met,Purchase date requirement is met,The sale price is too high,Not Applicable,Not Applicable,Not Applicable,5.303509e+10,23.0,PUGET SOUND ENERGY INC
3,Plug-in Hybrid Electric Vehicle (PHEV),KNDRJDJH5S,276871381,2025,KIA,Sorento,Passenger,30.0,0,Odometer reading is not collected at time of r...,...,False,Battery range requirement is met,This transaction type is not eligible for the ...,This transaction type is not eligible for the ...,Yes,Yes,No,5.303509e+10,23.0,PUGET SOUND ENERGY INC
4,Battery Electric Vehicle (BEV),5YJ3E1EA4J,280223583,2018,TESLA,Model 3,Passenger,215.0,"65,672",Actual Mileage,...,True,Battery range requirement is met,Purchase date requirement is met,Sale price/value requirement is met,Not Applicable,Not Applicable,Not Applicable,5.303303e+10,47.0,PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA)


In [7]:
# Data types and non-null counts
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1809203 entries, 0 to 1809202
Data columns (total 33 columns):
 #   Column                                                           Dtype  
---  ------                                                           -----  
 0   Clean Alternative Fuel Vehicle Type                              object 
 1   VIN (1-10)                                                       object 
 2   DOL Vehicle ID                                                   int64  
 3   Model Year                                                       int64  
 4   Make                                                             object 
 5   Model                                                            object 
 6   Primary Use                                                      object 
 7   Electric Range                                                   float64
 8   Odometer Reading                                                 object 
 9   Odometer Reading Descrip

In [10]:
# Missing values count
missing = df.isnull().sum().sort_values(ascending=False)
print('\nTop 15 columns with missing values:')
print(missing.head(15))


Top 15 columns with missing values:
Sale Date                                  1322235
Transportation Electrification Fee Paid      91031
Hybrid Vehicle Electrification Fee Paid      91031
Legislative District                          7468
Postal Code                                    120
2020 GEOID                                     120
County                                         120
Electric Range                                  26
State                                            1
Clean Alternative Fuel Vehicle Type              0
Primary Use                                      0
Odometer Reading                                 0
VIN (1-10)                                       0
Make                                             0
Model                                            0
dtype: int64


In [11]:
# Fill important missing values

# Numeric column
df['Electric Range'] = df['Electric Range'].fillna(df['Electric Range'].median())

# Text columns
df['County'] = df['County'].fillna('Unknown')
df['State'] = df['State'].fillna('Unknown')

# Postal code
df['Postal Code'] = df['Postal Code'].fillna(df['Postal Code'].mode()[0])

print('Important missing values handled successfully!')

print(df[['Electric Range', 'County', 'State', 'Postal Code']].isnull().sum())

Important missing values handled successfully!
Electric Range    0
County            0
State             0
Postal Code       0
dtype: int64


In [12]:
# Remove leading/trailing spaces from text columns
text_cols = df.select_dtypes(include='object').columns
df[text_cols] = df[text_cols].apply(lambda x: x.str.strip())

# Standardize important categorical columns
df['Make'] = df['Make'].str.title()
df['Model'] = df['Model'].str.title()
df['County'] = df['County'].str.title()
df['State'] = df['State'].str.upper()

print('Text standardization completed!')

# Check sample values
print(df[['Make', 'Model', 'County', 'State']].head())

Text standardization completed!
    Make    Model  County State
0  Tesla  Model 3    King    WA
1    Kia  Sorento  Kitsap    WA
2    Kia  Sorento  Kitsap    WA
3    Kia  Sorento  Kitsap    WA
4  Tesla  Model 3    King    WA


In [13]:
# Check duplicate rows
duplicate_count = df.duplicated().sum()

print('Duplicate rows:', duplicate_count)

Duplicate rows: 63


In [14]:
# Remove exact duplicate rows
before_rows = df.shape[0]

df = df.drop_duplicates()

after_rows = df.shape[0]

print('Rows before:', before_rows)
print('Rows after :', after_rows)
print('Duplicates removed:', before_rows - after_rows)

Rows before: 1809203
Rows after : 1809140
Duplicates removed: 63


In [15]:
# Standardize column names

df.columns = (
    df.columns
      .str.strip()                 # remove extra spaces
      .str.replace(' ', '_')      # space -> underscore
      .str.replace('/', '_')      # slash -> underscore
      .str.replace('-', '_')      # hyphen -> underscore
)

print('Column names standardized successfully!')

Column names standardized successfully!


In [16]:
# Show first 15 column names
print(df.columns[:15])

Index(['Clean_Alternative_Fuel_Vehicle_Type', 'VIN_(1_10)', 'DOL_Vehicle_ID',
       'Model_Year', 'Make', 'Model', 'Primary_Use', 'Electric_Range',
       'Odometer_Reading', 'Odometer_Reading_Description',
       'New_or_Used_Vehicle', 'Sale_Price', 'Sale_Date', 'Transaction_Type',
       'Transaction_Date'],
      dtype='object')


In [17]:
# Check invalid electric range values
negative_range = (df['Electric_Range'] < 0).sum()

# Check unrealistic model years
invalid_years = ((df['Model_Year'] < 1990) | (df['Model_Year'] > 2035)).sum()

print('Negative Electric Range values :', negative_range)
print('Unrealistic Model Year values  :', invalid_years)

# Quick summary
print('\nElectric Range summary:')
print(df['Electric_Range'].describe())

print('\nModel Year summary:')
print(df['Model_Year'].describe())

Negative Electric Range values : 0
Unrealistic Model Year values  : 0

Electric Range summary:
count    1.809140e+06
mean     6.338702e+01
std      9.046723e+01
min      0.000000e+00
25%      0.000000e+00
50%      1.900000e+01
75%      8.400000e+01
max      3.370000e+02
Name: Electric_Range, dtype: float64

Model Year summary:
count    1.809140e+06
mean     2.020199e+03
std      3.827547e+00
min      1.993000e+03
25%      2.018000e+03
50%      2.021000e+03
75%      2.023000e+03
max      2.027000e+03
Name: Model_Year, dtype: float64


In [23]:
import os

# Create database folder if it does not exist
os.makedirs('database', exist_ok=True)

print('Database folder ready!')

Database folder ready!


In [40]:
# Create SQLite database and load cleaned data

conn = sqlite3.connect('database/ev.db')

df.to_sql('registrations', conn, if_exists='replace', index=False)

# Verify table creation
check = pd.read_sql_query('SELECT COUNT(*) AS total_rows FROM registrations', conn)

print(check)



   total_rows
0     1809140


In [41]:
# Total registrations

query = '''
SELECT COUNT(*) AS total_registrations
FROM registrations;
'''

total_reg = pd.read_sql_query(query, conn)

print(total_reg)


   total_registrations
0              1809140


In [42]:
# Top manufacturers

query = '''
SELECT Make,
       COUNT(*) AS registrations
FROM registrations
GROUP BY Make
ORDER BY registrations DESC
LIMIT 12;
'''

top_make = pd.read_sql_query(query, conn)

print(top_make)


          Make  registrations
0        Tesla         721710
1       Nissan         205116
2    Chevrolet         154036
3         Ford         102216
4          Bmw          79422
5          Kia          77389
6       Toyota          75293
7      Hyundai          47012
8   Volkswagen          41629
9        Volvo          40334
10      Rivian          37399
11        Audi          35193


In [43]:
# Year-wise EV adoption trend

query = '''
SELECT Model_Year,
       COUNT(*) AS registrations
FROM registrations
WHERE Model_Year BETWEEN 2018 AND 2026
GROUP BY Model_Year
ORDER BY Model_Year;
'''

yearly_trend = pd.read_sql_query(query, conn)

print(yearly_trend)

   Model_Year  registrations
0        2018         150430
1        2019         103402
2        2020         102219
3        2021         150233
4        2022         183942
5        2023         302893
6        2024         195253
7        2025         107860
8        2026          64277


In [44]:
# Vehicle type distribution

query = '''
SELECT Clean_Alternative_Fuel_Vehicle_Type AS vehicle_type,
       COUNT(*) AS registrations
FROM registrations
GROUP BY Clean_Alternative_Fuel_Vehicle_Type
ORDER BY registrations DESC;
'''

vehicle_type = pd.read_sql_query(query, conn)

print(vehicle_type)

                             vehicle_type  registrations
0          Battery Electric Vehicle (BEV)        1394647
1  Plug-in Hybrid Electric Vehicle (PHEV)         414448
2                Hydrogen Powered Vehicle             45


In [45]:
# Top counties

query = '''
SELECT County,
       COUNT(*) AS registrations
FROM registrations
GROUP BY County
ORDER BY registrations DESC
LIMIT 10;
'''

top_counties = pd.read_sql_query(query, conn)

print(top_counties)

      County  registrations
0       King         927297
1  Snohomish         214535
2     Pierce         146370
3      Clark         106186
4     Kitsap          61875
5   Thurston          59763
6    Spokane          47740
7    Whatcom          44703
8     Benton          23751
9     Skagit          20175


In [46]:
# Average range by manufacturer

query = '''
SELECT Make,
       ROUND(AVG(Electric_Range), 2) AS avg_range,
       COUNT(*) AS registrations
FROM registrations
GROUP BY Make
HAVING COUNT(*) >= 200
ORDER BY avg_range DESC
LIMIT 10;
'''

avg_range = pd.read_sql_query(query, conn)

print(avg_range)

         Make  avg_range  registrations
0      Jaguar     205.09           2072
1       Tesla      90.18         721710
2   Chevrolet      86.95         154036
3        Fiat      82.50           9532
4      Nissan      80.51         205116
5       Smart      62.41           3540
6     Porsche      47.48          12085
7        Audi      46.56          35193
8         Kia      44.51          77389
9  Land Rover      42.04            980


In [47]:
# New vs used vehicles


query = '''
SELECT New_or_Used_Vehicle,
       COUNT(*) AS registrations
FROM registrations
GROUP BY New_or_Used_Vehicle
ORDER BY registrations DESC;
'''

new_used = pd.read_sql_query(query, conn)

print(new_used)

  New_or_Used_Vehicle  registrations
0                Used        1510017
1                 New         299123


In [48]:
# Top EV models

query = '''
SELECT Model,
       COUNT(*) AS registrations
FROM registrations
GROUP BY Model
ORDER BY registrations DESC
LIMIT 10;
'''

top_models = pd.read_sql_query(query, conn)

print(top_models)

                Model  registrations
0             Model Y         297368
1             Model 3         268871
2                Leaf         195252
3             Model S          93815
4                Volt          68570
5             Model X          54096
6             Bolt Ev          53998
7  Prius Prime (Phev)          29633
8      Mustang Mach-E          29450
9                Id.4          29391


In [49]:
# Fastest-growing manufacturers (2023-2024)

query = '''
WITH yearly AS (
    SELECT Make,
           Model_Year,
           COUNT(*) AS registrations
    FROM registrations
    WHERE Model_Year IN (2023, 2024)
    GROUP BY Make, Model_Year
)
SELECT Make,
       MAX(CASE WHEN Model_Year = 2023 THEN registrations END) AS reg_2023,
       MAX(CASE WHEN Model_Year = 2024 THEN registrations END) AS reg_2024,
       ROUND(
           (MAX(CASE WHEN Model_Year = 2024 THEN registrations END) -
            MAX(CASE WHEN Model_Year = 2023 THEN registrations END))
           * 100.0 /
           MAX(CASE WHEN Model_Year = 2023 THEN registrations END),
           2
       ) AS growth_pct
FROM yearly
GROUP BY Make
HAVING reg_2023 IS NOT NULL AND reg_2024 IS NOT NULL
ORDER BY growth_pct DESC
LIMIT 10;
'''

growth_make = pd.read_sql_query(query, conn)

print(growth_make)

         Make  reg_2023  reg_2024  growth_pct
0       Mazda         7      3683    52514.29
1    Cadillac       329      3705     1026.14
2      Jaguar        30        76      153.33
3      Toyota      5829     12761      118.92
4  Land Rover        39        76       94.87
5       Lexus      1191      2194       84.21
6        Audi      4020      6153       53.06
7         Kia     13524     14271        5.52
8      Subaru      4118      4323        4.98
9     Bentley         3         3        0.00


In [50]:
# Average range by vehicle type

query = '''
SELECT Clean_Alternative_Fuel_Vehicle_Type AS vehicle_type,
       ROUND(AVG(Electric_Range), 2) AS avg_range,
       COUNT(*) AS registrations
FROM registrations
GROUP BY Clean_Alternative_Fuel_Vehicle_Type
ORDER BY avg_range DESC;
'''

range_by_type = pd.read_sql_query(query, conn)

print(range_by_type)

                             vehicle_type  avg_range  registrations
0          Battery Electric Vehicle (BEV)      72.84        1394647
1  Plug-in Hybrid Electric Vehicle (PHEV)      31.57         414448
2                Hydrogen Powered Vehicle       0.00             45


In [51]:
# Top 3 manufacturer market share

query = '''
WITH make_counts AS (
    SELECT Make,
           COUNT(*) AS registrations
    FROM registrations
    GROUP BY Make
),
top3 AS (
    SELECT Make, registrations
    FROM make_counts
    ORDER BY registrations DESC
    LIMIT 3
),
total AS (
    SELECT COUNT(*) AS total_registrations
    FROM registrations
)
SELECT 
    Make,
    registrations,
    ROUND(
        registrations * 100.0 /
        (SELECT total_registrations FROM total),
        2
    ) AS market_share_pct
FROM top3
ORDER BY registrations DESC;
'''

top3_detail = pd.read_sql_query(query, conn)

print(top3_detail)

# Combined share
combined_share = top3_detail['market_share_pct'].sum()

print(f'\nCombined Top 3 Market Share: {combined_share:.2f}%')

        Make  registrations  market_share_pct
0      Tesla         721710             39.89
1     Nissan         205116             11.34
2  Chevrolet         154036              8.51

Combined Top 3 Market Share: 59.74%


In [52]:
# Final cleanup
conn.close()

print('SQL analysis completed and database connection closed.')

SQL analysis completed and database connection closed.


In [5]:
df.columns()

TypeError: 'Index' object is not callable